# Exercise: Building the GPT Model

Welcome! In this exercise, you will assemble the full GPT model from its core components. You've already learned about the individual pieces like the multi-head self-attention block and the MLP. Now it's time to put them all together!

In this exercise, you will:
*   Implement the `__init__` method of the `GPT` model, defining the token and positional embeddings, the stack of transformer `Block`s, and the final layer normalization and language modeling head.
*   Implement a custom weight initialization scheme, a common practice for training large language models.
*   Implement the `forward` pass, which defines how input data flows through the model to produce predictions.

Let's get started!

In [ ]:
import math
from dataclasses import dataclass

import torch
import torch.nn as nn
from torch.nn import functional as F

# --- Helper Code (Students should NOT modify this) ---

@dataclass
class GPTConfig:
    """Configuration class for the GPT model."""
    block_size: int = 256
    vocab_size: int = 65
    n_layer: int = 6
    n_head: int = 6
    n_embd: int = 384
    dropout: float = 0.2
    bias: bool = True # True: bias in Linears and LayerNorms, like GPT-2. False: a bit better and faster

class CausalSelfAttention(nn.Module):
    """A vanilla multi-head masked self-attention layer with a projection at the end."""
    def __init__(self, config: GPTConfig):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, bias=config.bias)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)
        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.dropout = config.dropout
        self.register_buffer("bias", torch.tril(torch.ones(config.block_size, config.block_size))
                                     .view(1, 1, config.block_size, config.block_size))

    def forward(self, x):
        B, T, C = x.size()
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
        att = att.masked_fill(self.bias[:,:,:T,:T] == 0, float('-inf'))
        att = F.softmax(att, dim=-1)
        att = self.attn_dropout(att)
        y = att @ v
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.resid_dropout(self.c_proj(y))
        return y

class MLP(nn.Module):
    """A simple MLP with one hidden layer and GELU activation."""
    def __init__(self, config: GPTConfig):
        super().__init__()
        self.c_fc = nn.Linear(config.n_embd, 4 * config.n_embd, bias=config.bias)
        self.gelu = nn.GELU()
        self.c_proj = nn.Linear(4 * config.n_embd, config.n_embd, bias=config.bias)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        x = self.c_fc(x)
        x = self.gelu(x)
        x = self.c_proj(x)
        x = self.dropout(x)
        return x

class Block(nn.Module):
    """A single Transformer block."""
    def __init__(self, config: GPTConfig):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.n_embd, bias=config.bias)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = nn.LayerNorm(config.n_embd, bias=config.bias)
        self.mlp = MLP(config)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

# --- End of Helper Code ---

# We will implement the GPT class in three parts:
# 1. The __init__ method
# 2. The _init_weights method
# 3. The forward method
class GPT(nn.Module):
    def __init__(self, config: GPTConfig):
        super().__init__()
        assert config.vocab_size is not None
        assert config.block_size is not None
        self.config = config
        
        # This is a dictionary to hold all the model's parameters.
        # It's a bit unusual but follows the nanoGPT repository's style.
        self.transformer = nn.ModuleDict() 
        # Call the weight initialization method
        self.apply(self._init_weights)

    def _init_weights(self, module):
        # This will be implemented in Exercise 2
        pass

    def forward(self, idx, targets=None):
        # This will be implemented in Exercise 3
        raise NotImplementedError

### Exercise 1: `__init__`

Your first task is to complete the `__init__` method for our `GPT` model. This involves creating all the necessary layers and storing them in the `self.transformer` `ModuleDict` and as top-level attributes.

Specifically, you need to define:
1.  `wte` (word token embedding): An `nn.Embedding` layer that maps vocabulary indices to `n_embd`-dimensional vectors.
2.  `wpe` (word position embedding): An `nn.Embedding` layer that maps position indices (from 0 to `block_size - 1`) to `n_embd`-dimensional vectors.
3.  `drop`: An `nn.Dropout` layer.
4.  `h`: A stack of `n_layer` `Block` modules, wrapped in an `nn.Sequential` container.
5.  `ln_f`: A final `nn.LayerNorm` layer.
6.  `lm_head`: The language modeling head, an `nn.Linear` layer that maps from `n_embd` to `vocab_size`.

**Hint:** Pay close attention to the input and output dimensions of each layer, which are defined in the `config` object. For the stack of `Block`s, a list comprehension inside `nn.Sequential` is a very clean way to create them.

In [ ]:
def gpt_init(self, config: GPTConfig):
    """
    Initializes the GPT model layers. This function will be assigned
    as the __init__ method of the GPT class for testing purposes.
    
    Args:
        self: The GPT model instance.
        config: A GPTConfig object with model hyperparameters.
    """
    super(GPT, self).__init__() # Important for PyTorch modules
    assert config.vocab_size is not None
    assert config.block_size is not None
    self.config = config
    
    ### START CODE HERE ###
    # your code here
    pass
    ### END CODE HERE ###

    # Report number of parameters
    print("number of parameters: %.2fM" % (self.get_num_params()/1e6,))

    # Initialize weights
    self.apply(self._init_weights)

# Helper function to get number of parameters
def get_num_params(self, non_embedding=True):
    n_params = sum(p.numel() for p in self.parameters())
    if non_embedding:
        n_params -= self.transformer.wpe.weight.numel()
    return n_params

# Assign the methods to the class
GPT.__init__ = gpt_init
GPT.get_num_params = get_num_params

In [ ]:
# Test for Exercise 1

# Let's create a small config for testing
test_config = GPTConfig(
    block_size=128,
    vocab_size=50,
    n_layer=4,
    n_head=4,
    n_embd=128,
    dropout=0.1
)

# Instantiate the model
model = GPT(test_config)
print(model)

# --- Assertions ---
# Check token embedding
wte = model.transformer.wte
assert isinstance(wte, nn.Embedding), "wte should be an nn.Embedding layer"
assert wte.num_embeddings == test_config.vocab_size, f"wte vocab size is wrong. Got {wte.num_embeddings}, expected {test_config.vocab_size}"
assert wte.embedding_dim == test_config.n_embd, f"wte embedding dim is wrong. Got {wte.embedding_dim}, expected {test_config.n_embd}"

# Check position embedding
wpe = model.transformer.wpe
assert isinstance(wpe, nn.Embedding), "wpe should be an nn.Embedding layer"
assert wpe.num_embeddings == test_config.block_size, f"wpe block size is wrong. Got {wpe.num_embeddings}, expected {test_config.block_size}"
assert wpe.embedding_dim == test_config.n_embd, f"wpe embedding dim is wrong. Got {wpe.embedding_dim}, expected {test_config.n_embd}"

# Check transformer blocks
h = model.transformer.h
assert isinstance(h, nn.Sequential), "transformer.h should be an nn.Sequential module"
assert len(h) == test_config.n_layer, f"Incorrect number of layers in transformer.h. Got {len(h)}, expected {test_config.n_layer}"
assert all(isinstance(block, Block) for block in h), "All elements in transformer.h should be Block instances"

# Check final layer norm
ln_f = model.transformer.ln_f
assert isinstance(ln_f, nn.LayerNorm), "transformer.ln_f should be an nn.LayerNorm layer"

# Check language model head
lm_head = model.lm_head
assert isinstance(lm_head, nn.Linear), "lm_head should be an nn.Linear layer"
assert lm_head.out_features == test_config.vocab_size, f"lm_head output features should be vocab_size. Got {lm_head.out_features}, expected {test_config.vocab_size}"
assert lm_head.in_features == test_config.n_embd, f"lm_head input features should be n_embd. Got {lm_head.in_features}, expected {test_config.n_embd}"

# Check for weight tying
assert torch.equal(model.transformer.wte.weight, model.lm_head.weight), "Weights of wte and lm_head should be tied (the same object)"

print("\n\033[92mAll tests for Exercise 1 passed!")

### Exercise 2: Weight Initialization

Great job! Now that the model structure is in place, let's implement the weight initialization. Proper initialization is crucial for stable training. For GPT-2, a specific scheme is used:
*   For `nn.Linear` layers, weights are initialized from a normal distribution with mean 0.0 and standard deviation 0.02. Biases are initialized to 0.
*   For `nn.Embedding` layers, weights are initialized similarly.
*   A special consideration from the GPT-2 paper: the weights of the projection layer in the MLP (`c_proj`) are scaled by `1 / sqrt(2 * n_layer)`. This helps control the variance of the activations as they pass through the residual connections in the network.

Your task is to implement the `_init_weights` method.

In [ ]:
def gpt_init_weights(self, module):
    """
    Initializes the weights of the model. This function will be assigned
    as the _init_weights method of the GPT class.
    
    Args:
        self: The GPT model instance.
        module: The module to initialize (PyTorch's .apply() will pass each module here).
    """
    ### START CODE HERE ###
    # your code here
    pass
    ### END CODE HERE ###

# Assign the method to the class
GPT._init_weights = gpt_init_weights

In [ ]:
# Test for Exercise 2

# Set a seed for reproducibility
torch.manual_seed(42)

# Re-initialize the model to trigger the _init_weights method via .apply()
model = GPT(test_config)

# --- Assertions ---
# Check a linear layer's bias
a_linear_layer_bias = model.transformer.h[0].mlp.c_fc.bias
assert a_linear_layer_bias is not None, "Bias should exist for this test"
assert torch.all(a_linear_layer_bias == 0), f"Linear layer biases should be initialized to zero. Got {a_linear_layer_bias}"

# Check a linear layer's weight standard deviation
a_linear_layer_weight = model.transformer.h[0].mlp.c_fc.weight
# The std of the sample should be close to the std of the distribution (0.02)
assert math.isclose(a_linear_layer_weight.std().item(), 0.02, abs_tol=1e-2), f"Linear layer weights should have std ~0.02. Got {a_linear_layer_weight.std().item()}"

# Check an embedding layer's weight standard deviation
embedding_layer_weight = model.transformer.wpe.weight
assert math.isclose(embedding_layer_weight.std().item(), 0.02, abs_tol=1e-2), f"Embedding layer weights should have std ~0.02. Got {embedding_layer_weight.std().item()}"

# Check the special scaling for the residual projection
# Expected std = 0.02 / sqrt(2 * n_layer)
n_layer = test_config.n_layer
expected_std = 0.02 / math.sqrt(2 * n_layer)
proj_weight = model.transformer.h[0].mlp.c_proj.weight
assert math.isclose(proj_weight.std().item(), expected_std, abs_tol=1e-3), f"Projection layer weights have incorrect std. Got {proj_weight.std().item()}, expected ~{expected_std}"

print("\n\033[92mAll tests for Exercise 2 passed!")

### Exercise 3: `forward`

You've reached the final and most important part: implementing the `forward` pass! This is where you define the flow of data through the model.

Here's the sequence of operations:
1.  Get the device of the input tensor `idx`. You'll need this to create the position indices tensor on the same device.
2.  Get token embeddings from `wte` and position embeddings from `wpe`.
3.  Combine them by simple addition. This is how the model learns about both the token's identity and its position.
4.  Pass the combined embeddings through the dropout layer, then the stack of transformer blocks (`h`), and finally the final layer norm (`ln_f`).
5.  If `targets` are provided (i.e., during training), calculate the cross-entropy loss between the model's output and the targets.
6.  If `targets` are `None` (i.e., during inference), simply pass the output of the transformer through the `lm_head` to get the final logits.

Good luck! This is the step that brings the whole model to life.

In [ ]:
def gpt_forward(self, idx, targets=None):
    """
    Performs the forward pass of the GPT model.
    
    Args:
        self: The GPT model instance.
        idx (torch.Tensor): A tensor of shape (B, T) with token indices.
        targets (torch.Tensor, optional): A tensor of shape (B, T) with target indices.
                                         If provided, the method also returns the loss.
                                         
    Returns:
        A tuple of (logits, loss) if targets are provided.
        A tuple of (logits, None) if targets are not provided.
    """
    B, T = idx.size()
    assert T <= self.config.block_size, f"Cannot forward sequence of length {T}, block size is only {self.config.block_size}"
    device = idx.device
    
    ### START CODE HERE ###
    # your code here
    pass
    ### END CODE HERE ###

    return logits, loss

# Assign the method to the class
GPT.forward = gpt_forward

In [ ]:
# Integration Test: Putting it all together

# Set a seed for reproducible random inputs
torch.manual_seed(1337)

# Use the same config and model as before
# The model now has __init__, _init_weights, and forward implemented
model = GPT(test_config)

# Create dummy input data
batch_size = 4
seq_len = 32
dummy_idx = torch.randint(0, test_config.vocab_size, (batch_size, seq_len))
dummy_targets = torch.randint(0, test_config.vocab_size, (batch_size, seq_len))

# --- Test Case 1: Forward pass with targets (training mode) ---
logits, loss = model(dummy_idx, dummy_targets)

# Check logits shape
expected_logits_shape = (batch_size, seq_len, test_config.vocab_size)
assert logits.shape == expected_logits_shape, f"Logits shape is incorrect. Got {logits.shape}, expected {expected_logits_shape}"

# Check loss
assert loss is not None, "Loss should be computed when targets are provided"
assert loss.ndim == 0, f"Loss should be a scalar tensor. Got shape {loss.shape}"

print("Test Case 1 (Training Mode) Passed!")
print(f"  Logits shape: {logits.shape}")
print(f"  Calculated loss: {loss.item():.4f}")

# --- Test Case 2: Forward pass without targets (inference mode) ---
logits_inf, loss_inf = model(dummy_idx)

# Check logits shape
assert logits_inf.shape == expected_logits_shape, f"Logits shape is incorrect in inference. Got {logits_inf.shape}, expected {expected_logits_shape}"

# Check loss (should be None)
assert loss_inf is None, "Loss should be None when targets are not provided"

print("\nTest Case 2 (Inference Mode) Passed!")
print(f"  Logits shape: {logits_inf.shape}")
print(f"  Loss: {loss_inf}")

print("\n\033[92mAll tests for Exercise 3 and integration passed! Congratulations on building a GPT model!")

### Challenge (Optional)

In the nanoGPT code, there is a helper method called `crop_block_size`. When loading a pre-trained model, you might want to use a different `block_size` (context window) than the one it was trained with. If the new `block_size` is smaller, the pre-trained positional embedding table (`wpe`) will be too large.

Your challenge is to implement a method `crop_block_size` that takes a new, smaller `block_size` and truncates the `wpe` weight tensor accordingly.

In [ ]:
def gpt_crop_block_size(self, block_size):
    """
    Crops the positional embedding table if the new block_size is smaller
    than the current one.
    
    Args:
        self: The GPT model instance.
        block_size (int): The new, smaller block size.
    """
    # superconfusing without the context of loading checkpoints, clean up later
    assert block_size <= self.config.block_size
    self.config.block_size = block_size
    ### START CODE HERE ###
    # your code here
    pass
    ### END CODE HERE ###

# Assign the challenge method to the class
GPT.crop_block_size = gpt_crop_block_size

# --- Test for Challenge ---
model = GPT(test_config)
original_wpe_shape = model.transformer.wpe.weight.shape
print(f"Original wpe shape: {original_wpe_shape}")

new_block_size = 64
model.crop_block_size(new_block_size)

cropped_wpe_shape = model.transformer.wpe.weight.shape
print(f"Cropped wpe shape: {cropped_wpe_shape}")

expected_shape = (new_block_size, test_config.n_embd)
assert cropped_wpe_shape == expected_shape, f"Cropped wpe shape is wrong. Got {cropped_wpe_shape}, expected {expected_shape}"

# Check that the model's config is also updated
assert model.config.block_size == new_block_size, "Model config was not updated"

print("\n\033[92mOptional Challenge Passed! You're ready to handle pre-trained models.")